# Phase 4 — Multi-Agent RAG

This notebook implements the **Multi-Agent RAG** route for the FYP.

The workflow follows the methodology:

**Original Question → Planning Agent → Iterative Hybrid Retrieval → Reranking → Answer Generation → Verification → Final Answer**

If verification says that the answer is not sufficiently supported, the system retrieves additional evidence or replans, then generates and verifies a revised answer.

## Agent collaboration design

The agents do **not** automatically know how to collaborate. Their behaviour is defined explicitly through:

1. **Role** — what the agent is responsible for.
2. **Input contract** — what information it receives.
3. **Rules** — what it is allowed / required to do.
4. **Output contract** — structured output passed to the next component.
5. **Shared state** — the object used to pass the plan, evidence, answer, verification feedback, and iteration history between agents.

This notebook uses:

- **Planning Agent — DeepSeek**
  - identifies information needs;
  - creates retrieval queries;
  - can revise a plan if verification requests replanning.

- **Retrieval Agent — BM25 + DPR/FAISS**
  - executes the planner's retrieval queries;
  - combines sparse and dense candidates;
  - preserves retrieval provenance;
  - removes duplicate passages.

- **Reranking Agent — `BAAI/bge-reranker-large`**
  - scores the accumulated evidence against the original question;
  - keeps the most relevant Top-10 passages.

- **Answer Agent — DeepSeek**
  - generates a short grounded answer from the plan and selected evidence.

- **Verification Agent — DeepSeek**
  - checks whether the answer is supported by the selected evidence;
  - returns `SUPPORTED`, `RETRIEVE_MORE`, or `REPLAN`;
  - supplies follow-up retrieval queries when more evidence is required.

The LLM agents can use the **same DeepSeek model** with different role-specific prompts. They do not need separate fine-tuned models.

## Step 0 — Keep the Development Setting Consistent

For a fair development comparison, this notebook defaults to the **same first 20 BeerQA questions** and the same `beerqa_dev20` Wikipedia corpus used by Standard RAG.

If you later increase the development sample to 40, create matching:

- `beerqa_dev40_wikipedia_corpus.jsonl`
- `beerqa_dev40_dpr_embeddings.npy`
- `beerqa_dev40.faiss`

and evaluate **Standard RAG, Query Decomposition RAG, and Multi-Agent RAG on the same 40 questions**.

## Step 1 — Imports and Configuration

In [1]:
from pathlib import Path
from collections import Counter
import os
import json
import re
import string
import getpass

import numpy as np
import torch
import faiss

from rank_bm25 import BM25Okapi

from transformers import (
    DPRQuestionEncoder,
    DPRQuestionEncoderTokenizer,
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

from openai import OpenAI

C:\Users\Wei Khang\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ------------------------------------------------------------
# Development configuration
# ------------------------------------------------------------

DEV_SAMPLE_SIZE = 20
CORPUS_TAG = f"beerqa_dev{DEV_SAMPLE_SIZE}"

OUTPUT_DIR = Path(
    r"C:\Users\Wei Khang\Documents\Master\FYP\processed_data"
)

BEERQA_DEV_PATH = Path(
    r"C:\Users\Wei Khang\Documents\Master\FYP\beerqa_dev_v1.0.json"
)

BEERQA_CORPUS_PATH = (
    OUTPUT_DIR /
    f"{CORPUS_TAG}_wikipedia_corpus.jsonl"
)

BEERQA_DPR_EMBEDDINGS_PATH = (
    OUTPUT_DIR /
    f"{CORPUS_TAG}_dpr_embeddings.npy"
)

BEERQA_FAISS_PATH = (
    OUTPUT_DIR /
    f"{CORPUS_TAG}.faiss"
)

STANDARD_RAG_EVAL_PATH = (
    OUTPUT_DIR /
    f"standard_rag_dev{DEV_SAMPLE_SIZE}_evaluated.jsonl"
)

QD_RAG_EVAL_PATH = (
    OUTPUT_DIR /
    f"query_decomposition_rag_dev{DEV_SAMPLE_SIZE}_evaluated.jsonl"
)

MA_RAG_DEV5_RESULTS_PATH = (
    OUTPUT_DIR /
    "multi_agent_rag_dev5_results.jsonl"
)

MA_RAG_DEV20_RESULTS_PATH = (
    OUTPUT_DIR /
    f"multi_agent_rag_dev{DEV_SAMPLE_SIZE}_results.jsonl"
)

MA_RAG_DEV20_EVAL_PATH = (
    OUTPUT_DIR /
    f"multi_agent_rag_dev{DEV_SAMPLE_SIZE}_evaluated.jsonl"
)

print("Development questions:", DEV_SAMPLE_SIZE)
print("BeerQA dev exists:", BEERQA_DEV_PATH.exists())
print("Corpus exists:", BEERQA_CORPUS_PATH.exists())
print("Embeddings exist:", BEERQA_DPR_EMBEDDINGS_PATH.exists())
print("FAISS exists:", BEERQA_FAISS_PATH.exists())

if not BEERQA_CORPUS_PATH.exists():
    raise FileNotFoundError(
        f"{BEERQA_CORPUS_PATH.name} is missing. "
        "Build the matching BeerQA development corpus first."
    )

Development questions: 20
BeerQA dev exists: True
Corpus exists: True
Embeddings exist: True
FAISS exists: True


## Step 2 — Load BeerQA Dev Sample and Shared Wikipedia Corpus

In [3]:
def normalize_title(title):
    return " ".join(
        str(title)
        .replace("_", " ")
        .strip()
        .lower()
        .split()
    )


def bm25_tokenize(text):
    return re.findall(
        r"\b\w+\b",
        str(text).lower()
    )

In [4]:
with open(
    BEERQA_DEV_PATH,
    "r",
    encoding="utf-8"
) as f:
    beerqa_dev = json.load(f)

beerqa_records = beerqa_dev["data"]
beerqa_sample = beerqa_records[:DEV_SAMPLE_SIZE]

print("Total BeerQA dev records:", len(beerqa_records))
print("Development sample:", len(beerqa_sample))

Total BeerQA dev records: 14121
Development sample: 20


In [5]:
beerqa_corpus = []

with open(
    BEERQA_CORPUS_PATH,
    "r",
    encoding="utf-8"
) as f:
    for line in f:
        beerqa_corpus.append(
            json.loads(line)
        )

passage_ids = [
    item["passage_id"]
    for item in beerqa_corpus
]

word_counts = [
    len(item["text"].split())
    for item in beerqa_corpus
]

print("Loaded passages:", len(beerqa_corpus))
print("Unique passage IDs:", len(set(passage_ids)))
print("Maximum chunk words:", max(word_counts))

assert len(passage_ids) == len(set(passage_ids))
assert max(word_counts) <= 100

print("PASS: shared BeerQA corpus is ready.")

Loaded passages: 1674
Unique passage IDs: 1674
Maximum chunk words: 100
PASS: shared BeerQA corpus is ready.


## Step 3 — Build the Shared BM25 Retriever

In [6]:
beerqa_bm25_documents = [
    chunk["title"] + " " + chunk["text"]
    for chunk in beerqa_corpus
]

beerqa_tokenized_corpus = [
    bm25_tokenize(document)
    for document in beerqa_bm25_documents
]

beerqa_bm25 = BM25Okapi(
    beerqa_tokenized_corpus
)

print(
    "BM25 ready:",
    len(beerqa_tokenized_corpus),
    "passages"
)

BM25 ready: 1674 passages


In [7]:
def beerqa_bm25_search(
    query,
    top_k=10
):
    tokenized_query = bm25_tokenize(query)

    scores = beerqa_bm25.get_scores(
        tokenized_query
    )

    top_indices = np.argsort(
        scores
    )[::-1][:top_k]

    results = []

    for rank, idx in enumerate(
        top_indices,
        start=1
    ):
        chunk = beerqa_corpus[idx]

        results.append({
            "rank": rank,
            "passage_id": chunk["passage_id"],
            "article_id": chunk["article_id"],
            "title": chunk["title"],
            "text": chunk["text"],
            "bm25_score": float(scores[idx]),
        })

    return results

## Step 4 — Load DPR Question Encoder, Embeddings, and FAISS

In [8]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

QUESTION_ENCODER_NAME = (
    "facebook/dpr-question_encoder-multiset-base"
)

question_tokenizer = (
    DPRQuestionEncoderTokenizer
    .from_pretrained(
        QUESTION_ENCODER_NAME
    )
)

question_encoder = (
    DPRQuestionEncoder
    .from_pretrained(
        QUESTION_ENCODER_NAME
    )
    .to(device)
)

question_encoder.eval()

print("DPR question encoder loaded.")

Device: cpu


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 28112.34it/s]
[transformers] DPRQuestionEncoder LOAD REPORT from: facebook/dpr-question_encoder-multiset-base
Key                                             | Status     |  | 
------------------------------------------------+------------+--+-
question_encoder.bert_model.pooler.dense.weight | UNEXPECTED |  | 
question_encoder.bert_model.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DPR question encoder loaded.


In [9]:
if not BEERQA_DPR_EMBEDDINGS_PATH.exists():
    raise FileNotFoundError(
        "Matching DPR passage embeddings are missing."
    )

beerqa_passage_embeddings = np.load(
    BEERQA_DPR_EMBEDDINGS_PATH
)

print(
    "Loaded DPR embeddings:",
    beerqa_passage_embeddings.shape
)

assert (
    beerqa_passage_embeddings.shape[0]
    ==
    len(beerqa_corpus)
)

assert (
    beerqa_passage_embeddings.shape[1]
    ==
    768
)

print("PASS: embeddings match corpus.")

Loaded DPR embeddings: (1674, 768)
PASS: embeddings match corpus.


In [10]:
if BEERQA_FAISS_PATH.exists():

    beerqa_faiss_index = faiss.read_index(
        str(BEERQA_FAISS_PATH)
    )

else:

    print("FAISS file missing. Rebuilding from saved embeddings...")

    beerqa_faiss_index = faiss.IndexFlatIP(
        beerqa_passage_embeddings.shape[1]
    )

    beerqa_faiss_index.add(
        beerqa_passage_embeddings.astype(
            "float32"
        )
    )

    faiss.write_index(
        beerqa_faiss_index,
        str(BEERQA_FAISS_PATH)
    )

print(
    "FAISS vectors:",
    beerqa_faiss_index.ntotal
)

assert (
    beerqa_faiss_index.ntotal
    ==
    len(beerqa_corpus)
)

print("PASS: FAISS matches corpus.")

FAISS vectors: 1674
PASS: FAISS matches corpus.


In [11]:
def beerqa_dpr_search(
    query,
    top_k=10
):
    inputs = question_tokenizer(
        query,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    inputs = {
        key: value.to(device)
        for key, value
        in inputs.items()
    }

    with torch.inference_mode():

        query_embedding = (
            question_encoder(**inputs)
            .pooler_output
            .cpu()
            .numpy()
            .astype("float32")
        )

    scores, indices = (
        beerqa_faiss_index.search(
            query_embedding,
            top_k
        )
    )

    results = []

    for rank, (idx, score) in enumerate(
        zip(indices[0], scores[0]),
        start=1
    ):
        chunk = beerqa_corpus[idx]

        results.append({
            "rank": rank,
            "passage_id": chunk["passage_id"],
            "article_id": chunk["article_id"],
            "title": chunk["title"],
            "text": chunk["text"],
            "dpr_score": float(score),
        })

    return results

## Step 5 — Load the BGE Reranker

The reranker is a shared deterministic component in the multi-agent workflow.

It is treated as the **Reranking Agent** because it has a specialized responsibility:
**score accumulated candidate evidence and retain the most relevant passages**.

If Hugging Face model loading fails, inspect the full `OSError` before changing the architecture.

In [12]:
RERANKER_MODEL_NAME = (
    "BAAI/bge-reranker-large"
)

print("Loading reranker tokenizer...")

reranker_tokenizer = (
    AutoTokenizer.from_pretrained(
        RERANKER_MODEL_NAME
    )
)

print("Loading reranker model...")

reranker_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        RERANKER_MODEL_NAME
    )
    .to(device)
)

reranker_model.eval()

print(
    "Reranker loaded:",
    RERANKER_MODEL_NAME
)

Loading reranker tokenizer...
Loading reranker model...


Loading weights: 100%|██████████| 393/393 [00:01<00:00, 229.88it/s]

Reranker loaded: BAAI/bge-reranker-large


In [13]:
def rerank_candidates(
    query,
    candidates,
    top_k=10,
    batch_size=4
):
    if not candidates:
        return []

    passages = [
        f"{candidate['title']}. "
        f"{candidate['text']}"
        for candidate in candidates
    ]

    all_scores = []

    for start in range(
        0,
        len(passages),
        batch_size
    ):
        batch_passages = passages[
            start:start + batch_size
        ]

        batch_queries = [
            query
        ] * len(batch_passages)

        inputs = reranker_tokenizer(
            batch_queries,
            batch_passages,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )

        inputs = {
            key: value.to(device)
            for key, value
            in inputs.items()
        }

        with torch.inference_mode():

            outputs = reranker_model(
                **inputs
            )

            scores = (
                outputs.logits
                .view(-1)
                .float()
                .cpu()
                .tolist()
            )

        all_scores.extend(scores)

    reranked = []

    for candidate, score in zip(
        candidates,
        all_scores
    ):
        result = candidate.copy()

        result["reranker_score"] = (
            float(score)
        )

        reranked.append(result)

    reranked.sort(
        key=lambda x:
            x["reranker_score"],
        reverse=True
    )

    return reranked[:top_k]

## Step 6 — Connect DeepSeek

In [14]:
if not os.getenv(
    "DEEPSEEK_API_KEY"
):
    os.environ[
        "DEEPSEEK_API_KEY"
    ] = getpass.getpass(
        "Enter your DeepSeek API key: "
    )

client = OpenAI(
    api_key=os.environ[
        "DEEPSEEK_API_KEY"
    ],
    base_url="https://api.deepseek.com"
)

DEEPSEEK_MODEL = (
    "deepseek-v4-flash"
)

print("DeepSeek client ready.")

DeepSeek client ready.


# Multi-Agent Roles and Rules

## Step 7 — Define Explicit Agent Contracts

Yes — this is the part you manually design.

You do **not** need to train five different models. The role specialization comes from:
- the prompt/rules;
- the input each agent receives;
- the output schema;
- the shared state used to hand work to the next component.

The following dictionary documents the responsibilities used in this implementation.

In [15]:
AGENT_RULES = {
    "planning_agent": {
        "purpose":
            "Identify the minimum information needs and retrieval queries required to answer the question.",
        "must_do": [
            "Preserve important entity names.",
            "Create independently searchable retrieval queries.",
            "Represent dependencies between information needs when useful.",
            "Do not answer the question.",
            "Return structured JSON only.",
        ],
    },

    "retrieval_agent": {
        "purpose":
            "Execute hybrid BM25 + DPR/FAISS retrieval for planner or verifier queries.",
        "must_do": [
            "Search the shared Wikipedia corpus only.",
            "Run both BM25 and DPR for every retrieval query.",
            "Merge candidates by passage_id.",
            "Preserve which query and retriever found each passage.",
            "Do not generate the final answer.",
        ],
    },

    "reranking_agent": {
        "purpose":
            "Prioritize accumulated evidence using the cross-encoder reranker.",
        "must_do": [
            "Score evidence against the original question.",
            "Keep only the final Top-k passages.",
            "Do not modify passage text.",
        ],
    },

    "answer_agent": {
        "purpose":
            "Generate the final factual answer from selected evidence.",
        "must_do": [
            "Use only retrieved evidence.",
            "Follow the planner's information requirements.",
            "Return a short factual answer.",
            "Return INSUFFICIENT_EVIDENCE when evidence is inadequate.",
        ],
    },

    "verification_agent": {
        "purpose":
            "Check whether the generated answer is supported and the plan has been addressed.",
        "must_do": [
            "Use only the supplied evidence.",
            "Return SUPPORTED, RETRIEVE_MORE, or REPLAN.",
            "When more evidence is needed, provide concrete follow-up retrieval queries.",
            "Do not silently correct the answer using outside knowledge.",
            "Return structured JSON only.",
        ],
    },
}

for agent_name, rules in AGENT_RULES.items():
    print("=" * 80)
    print(agent_name)
    print("Purpose:", rules["purpose"])
    for rule in rules["must_do"]:
        print("-", rule)

planning_agent
Purpose: Identify the minimum information needs and retrieval queries required to answer the question.
- Preserve important entity names.
- Create independently searchable retrieval queries.
- Represent dependencies between information needs when useful.
- Do not answer the question.
- Return structured JSON only.
retrieval_agent
Purpose: Execute hybrid BM25 + DPR/FAISS retrieval for planner or verifier queries.
- Search the shared Wikipedia corpus only.
- Run both BM25 and DPR for every retrieval query.
- Merge candidates by passage_id.
- Preserve which query and retriever found each passage.
- Do not generate the final answer.
reranking_agent
Purpose: Prioritize accumulated evidence using the cross-encoder reranker.
- Score evidence against the original question.
- Keep only the final Top-k passages.
- Do not modify passage text.
answer_agent
Purpose: Generate the final factual answer from selected evidence.
- Use only retrieved evidence.
- Follow the planner's informa

## Step 8 — Define the Shared Collaboration State

The agents collaborate by reading/writing a shared state.

This is more reproducible than letting agents send unrestricted free-form messages to each other.

In [16]:
def create_multi_agent_state(
    question
):
    return {
        "question":
            question,

        "plan":
            None,

        "current_queries":
            [],

        "candidate_pool":
            {},

        "reranked_evidence":
            [],

        "answer":
            None,

        "verification":
            None,

        "iteration":
            0,

        "history":
            [],
    }

## Step 9 — Planning Agent

The Planning Agent does not produce a detailed hidden chain-of-thought.

It returns only the **task-level plan needed for retrieval**:
- information needs;
- retrieval queries;
- optional dependencies;
- short answer strategy.

In [17]:
def _strip_json_fence(
    text
):
    cleaned = str(text).strip()

    cleaned = re.sub(
        r"^```(?:json)?\s*",
        "",
        cleaned,
        flags=re.IGNORECASE
    )

    cleaned = re.sub(
        r"\s*```$",
        "",
        cleaned
    )

    return cleaned.strip()


def planner_agent(
    question,
    verification_feedback=None,
    model=DEEPSEEK_MODEL,
    max_steps=4
):
    system_prompt = """
You are the Planning Agent in a Multi-Agent RAG system for
open-domain multi-hop question answering.

Your job is to identify the minimum information requirements needed
to answer the question and convert them into independently searchable
Wikipedia retrieval queries.

Rules:
1. Do not answer the original question.
2. Preserve important entity names exactly.
3. Use between 1 and 4 plan steps.
4. Each step must contain one clear information need and one concise
   retrieval query.
5. Add dependencies only when a later step logically depends on an
   earlier information need.
6. If verification feedback is provided, revise the plan only where
   necessary.
7. Return ONLY valid JSON.

Required format:
{
  "steps": [
    {
      "step_id": 1,
      "information_need": "...",
      "retrieval_query": "...",
      "depends_on": []
    }
  ],
  "answer_strategy": "short description"
}
""".strip()

    feedback_text = (
        "None"
        if verification_feedback is None
        else json.dumps(
            verification_feedback,
            ensure_ascii=False
        )
    )

    user_prompt = f"""
Original Question:
{question}

Previous Verification Feedback:
{feedback_text}

Create the retrieval plan JSON only.
""".strip()

    response = (
        client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": user_prompt
                }
            ],
            max_tokens=450,
            temperature=0.0,
            extra_body={
                "thinking": {
                    "type": "disabled"
                }
            }
        )
    )

    raw_output = (
        response
        .choices[0]
        .message
        .content
        .strip()
    )

    cleaned = _strip_json_fence(
        raw_output
    )

    used_fallback = False

    try:
        parsed = json.loads(cleaned)

        steps = parsed.get(
            "steps",
            []
        )

        answer_strategy = (
            parsed.get(
                "answer_strategy",
                ""
            )
        )

    except Exception:
        steps = []
        answer_strategy = ""

    valid_steps = []

    for step in steps[:max_steps]:

        if not isinstance(
            step,
            dict
        ):
            continue

        retrieval_query = " ".join(
            str(
                step.get(
                    "retrieval_query",
                    ""
                )
            ).split()
        )

        information_need = " ".join(
            str(
                step.get(
                    "information_need",
                    ""
                )
            ).split()
        )

        if not retrieval_query:
            continue

        valid_steps.append({
            "step_id":
                len(valid_steps) + 1,

            "information_need":
                information_need
                or retrieval_query,

            "retrieval_query":
                retrieval_query,

            "depends_on":
                step.get(
                    "depends_on",
                    []
                ),
        })

    if not valid_steps:

        used_fallback = True

        valid_steps = [{
            "step_id":
                1,

            "information_need":
                "Retrieve evidence needed to answer the original question.",

            "retrieval_query":
                question,

            "depends_on":
                [],
        }]

        answer_strategy = (
            "Use retrieved evidence to answer the original question."
        )

    return {
        "steps":
            valid_steps,

        "answer_strategy":
            answer_strategy,

        "raw_output":
            raw_output,

        "used_fallback":
            used_fallback,
    }

### Step 9A — Test the Planning Agent on One BeerQA Question

In [19]:
test_record = beerqa_sample[0]
test_question = test_record["question"]

print("Question:")
print(test_question)

print("\nGold:")
print(test_record["answers"])

planning_test = planner_agent(
    test_question
)

print("\nPlan:")

for step in planning_test["steps"]:
    print(
        step["step_id"],
        "| Need:",
        step["information_need"]
    )
    print(
        "  Query:",
        step["retrieval_query"]
    )
    print(
        "  Depends on:",
        step["depends_on"]
    )

print(
    "\nFallback used:",
    planning_test["used_fallback"]
)

Question:
Which genus contains more species, Ortegocactus or Eschscholzia?

Gold:
['Eschscholzia']


APIStatusError: Error code: 402 - {'error': {'message': 'Insufficient Balance (request_id: f18ea2eb-af15-4288-b013-15e5e26f1950)', 'type': 'insufficient_balance_error', 'param': None, 'code': 'invalid_request_error'}}

## Step 10 — Retrieval Agent

The Retrieval Agent is deterministic:
- run BM25 Top-k;
- run DPR/FAISS Top-k;
- merge by `passage_id`;
- attach provenance;
- accumulate evidence across iterations.

In [ ]:
def hybrid_candidates_for_query(
    query,
    top_k_each=10
):
    bm25_results = (
        beerqa_bm25_search(
            query,
            top_k=top_k_each
        )
    )

    dpr_results = (
        beerqa_dpr_search(
            query,
            top_k=top_k_each
        )
    )

    combined = {}

    for result in bm25_results:

        pid = result["passage_id"]

        if pid not in combined:
            combined[pid] = {
                "passage_id":
                    pid,

                "article_id":
                    result["article_id"],

                "title":
                    result["title"],

                "text":
                    result["text"],

                "bm25_rank":
                    None,

                "dpr_rank":
                    None,

                "bm25_score":
                    None,

                "dpr_score":
                    None,

                "source":
                    [],
            }

        combined[pid][
            "bm25_rank"
        ] = result["rank"]

        combined[pid][
            "bm25_score"
        ] = result[
            "bm25_score"
        ]

        combined[pid][
            "source"
        ].append("BM25")

    for result in dpr_results:

        pid = result[
            "passage_id"
        ]

        if pid not in combined:
            combined[pid] = {
                "passage_id":
                    pid,

                "article_id":
                    result["article_id"],

                "title":
                    result["title"],

                "text":
                    result["text"],

                "bm25_rank":
                    None,

                "dpr_rank":
                    None,

                "bm25_score":
                    None,

                "dpr_score":
                    None,

                "source":
                    [],
            }

        combined[pid][
            "dpr_rank"
        ] = result["rank"]

        combined[pid][
            "dpr_score"
        ] = result[
            "dpr_score"
        ]

        if (
            "DPR"
            not in
            combined[pid]["source"]
        ):
            combined[pid][
                "source"
            ].append("DPR")

    return list(
        combined.values()
    )

In [ ]:
def retrieval_agent(
    state,
    queries,
    top_k_each=10
):
    newly_seen = 0

    for query_index, query in enumerate(
        queries,
        start=1
    ):
        candidates = (
            hybrid_candidates_for_query(
                query=query,
                top_k_each=top_k_each
            )
        )

        for candidate in candidates:

            pid = candidate[
                "passage_id"
            ]

            if (
                pid
                not in
                state[
                    "candidate_pool"
                ]
            ):
                state[
                    "candidate_pool"
                ][pid] = {
                    "passage_id":
                        candidate[
                            "passage_id"
                        ],

                    "article_id":
                        candidate[
                            "article_id"
                        ],

                    "title":
                        candidate[
                            "title"
                        ],

                    "text":
                        candidate[
                            "text"
                        ],

                    "retrieval_history":
                        [],
                }

                newly_seen += 1

            state[
                "candidate_pool"
            ][pid][
                "retrieval_history"
            ].append({
                "iteration":
                    state[
                        "iteration"
                    ],

                "query_index":
                    query_index,

                "query":
                    query,

                "source":
                    candidate[
                        "source"
                    ],

                "bm25_rank":
                    candidate[
                        "bm25_rank"
                    ],

                "dpr_rank":
                    candidate[
                        "dpr_rank"
                    ],

                "bm25_score":
                    candidate[
                        "bm25_score"
                    ],

                "dpr_score":
                    candidate[
                        "dpr_score"
                    ],
            })

    state["history"].append({
        "event":
            "retrieval",

        "iteration":
            state[
                "iteration"
            ],

        "queries":
            list(
                queries
            ),

        "new_candidates":
            newly_seen,

        "total_candidates":
            len(
                state[
                    "candidate_pool"
                ]
            ),
    })

    return state

### Step 10A — Test Planning → Retrieval Collaboration

In [ ]:
state_test = create_multi_agent_state(
    test_question
)

state_test["plan"] = planning_test

state_test["current_queries"] = [
    step["retrieval_query"]
    for step in planning_test[
        "steps"
    ]
]

state_test["iteration"] = 1

state_test = retrieval_agent(
    state=state_test,
    queries=state_test[
        "current_queries"
    ],
    top_k_each=10
)

print(
    "Planner queries:",
    state_test[
        "current_queries"
    ]
)

print(
    "Unique accumulated candidates:",
    len(
        state_test[
            "candidate_pool"
        ]
    )
)

assert (
    len(
        state_test[
            "candidate_pool"
        ]
    )
    ==
    len(
        set(
            state_test[
                "candidate_pool"
            ].keys()
        )
    )
)

print(
    "PASS: retrieval candidate pool has no duplicate passage IDs."
)

## Step 11 — Reranking Agent

The Reranking Agent evaluates the entire accumulated candidate pool against the **original question**.

In [ ]:
def reranking_agent(
    state,
    final_top_k=10,
    batch_size=4
):
    candidates = list(
        state[
            "candidate_pool"
        ].values()
    )

    reranked = rerank_candidates(
        query=state[
            "question"
        ],
        candidates=candidates,
        top_k=final_top_k,
        batch_size=batch_size
    )

    state[
        "reranked_evidence"
    ] = reranked

    state["history"].append({
        "event":
            "reranking",

        "iteration":
            state[
                "iteration"
            ],

        "input_candidates":
            len(
                candidates
            ),

        "selected_evidence":
            len(
                reranked
            ),
    })

    return state

In [ ]:
state_test = reranking_agent(
    state=state_test,
    final_top_k=10
)

print(
    "Selected evidence:",
    len(
        state_test[
            "reranked_evidence"
        ]
    )
)

for rank, passage in enumerate(
    state_test[
        "reranked_evidence"
    ],
    start=1
):
    print(
        rank,
        "|",
        passage[
            "passage_id"
        ],
        "|",
        passage[
            "title"
        ],
        "|",
        round(
            passage[
                "reranker_score"
            ],
            4
        )
    )

## Step 12 — Answer Agent

The Answer Agent receives:
- original question;
- concise planner information needs;
- reranked evidence.

It does not see BeerQA gold answers.

In [ ]:
def build_agent_evidence_context(
    evidence,
    max_passages=10
):
    blocks = []

    for rank, passage in enumerate(
        evidence[
            :max_passages
        ],
        start=1
    ):
        blocks.append(
            f"[Passage {rank}]\n"
            f"Title: {passage['title']}\n"
            f"Text: {passage['text']}"
        )

    return "\n\n".join(
        blocks
    )


def answer_agent(
    state,
    model=DEEPSEEK_MODEL
):
    information_needs = "\n".join(
        f"- {step['information_need']}"
        for step in state[
            "plan"
        ]["steps"]
    )

    evidence_context = (
        build_agent_evidence_context(
            state[
                "reranked_evidence"
            ]
        )
    )

    system_prompt = """
You are the Answer Agent in a Multi-Agent RAG system.

Generate the final answer to the original question using ONLY the
retrieved Wikipedia evidence.

Rules:
1. Use the planner's information needs as a checklist.
2. Do not use unsupported outside knowledge.
3. Connect evidence from multiple passages when required.
4. Return only the short factual answer.
5. Do not explain agent collaboration or intermediate reasoning.
6. If the evidence is insufficient, return:
INSUFFICIENT_EVIDENCE
""".strip()

    user_prompt = f"""
Original Question:
{state['question']}

Planner Information Needs:
{information_needs}

Retrieved Wikipedia Evidence:
{evidence_context}

Final Answer:
""".strip()

    response = (
        client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role":
                        "system",

                    "content":
                        system_prompt
                },
                {
                    "role":
                        "user",

                    "content":
                        user_prompt
                }
            ],
            max_tokens=100,
            temperature=0.0,
            extra_body={
                "thinking": {
                    "type":
                        "disabled"
                }
            }
        )
    )

    state["answer"] = (
        response
        .choices[0]
        .message
        .content
        .strip()
    )

    state["history"].append({
        "event":
            "answer_generation",

        "iteration":
            state[
                "iteration"
            ],

        "answer":
            state[
                "answer"
            ],
    })

    return state

### Step 12A — Test the Answer Agent

In [ ]:
state_test = answer_agent(
    state_test
)

print("Question:")
print(state_test["question"])

print("\nGenerated Answer:")
print(state_test["answer"])

print("\nGold Answer (for development inspection only):")
print(test_record["answers"])

## Step 13 — Verification Agent

The Verification Agent must not use BeerQA gold answers.

It judges whether:
- the generated answer is supported by the retrieved passages;
- the planner's information requirements are sufficiently addressed.

It returns one of:

- `SUPPORTED`
- `RETRIEVE_MORE`
- `REPLAN`

`RETRIEVE_MORE` should include one or more follow-up retrieval queries.

In [ ]:
def verifier_agent(
    state,
    model=DEEPSEEK_MODEL,
    max_follow_up_queries=3
):
    information_needs = [
        step["information_need"]
        for step in state[
            "plan"
        ]["steps"]
    ]

    evidence_context = (
        build_agent_evidence_context(
            state[
                "reranked_evidence"
            ]
        )
    )

    system_prompt = """
You are the Verification Agent in a Multi-Agent RAG system.

Evaluate whether the proposed answer is sufficiently supported by
the supplied Wikipedia evidence and whether the planner's information
needs have been addressed.

You must use ONLY the supplied evidence.

Choose exactly one status:

SUPPORTED
- The answer directly addresses the question and is sufficiently
  supported by the evidence.

RETRIEVE_MORE
- The plan is reasonable, but important supporting evidence is
  missing. Provide concrete follow-up Wikipedia retrieval queries.

REPLAN
- The current plan missed or misunderstood an important information
  requirement. Explain the missing requirement briefly so the
  Planning Agent can revise the plan.

Rules:
1. Do not answer the question yourself.
2. Do not use outside knowledge.
3. Keep feedback concise.
4. Return ONLY valid JSON.

Required format:
{
  "status": "SUPPORTED | RETRIEVE_MORE | REPLAN",
  "reason": "brief reason",
  "missing_information": ["..."],
  "follow_up_queries": ["..."]
}
""".strip()

    user_prompt = f"""
Original Question:
{state['question']}

Planner Information Needs:
{json.dumps(information_needs, ensure_ascii=False)}

Proposed Answer:
{state['answer']}

Retrieved Evidence:
{evidence_context}

Return verification JSON only.
""".strip()

    response = (
        client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role":
                        "system",

                    "content":
                        system_prompt
                },
                {
                    "role":
                        "user",

                    "content":
                        user_prompt
                }
            ],
            max_tokens=350,
            temperature=0.0,
            extra_body={
                "thinking": {
                    "type":
                        "disabled"
                }
            }
        )
    )

    raw_output = (
        response
        .choices[0]
        .message
        .content
        .strip()
    )

    cleaned = _strip_json_fence(
        raw_output
    )

    try:
        parsed = json.loads(
            cleaned
        )
    except Exception:
        parsed = {}

    status = str(
        parsed.get(
            "status",
            ""
        )
    ).strip().upper()

    valid_statuses = {
        "SUPPORTED",
        "RETRIEVE_MORE",
        "REPLAN",
    }

    used_fallback = False

    if (
        status
        not in
        valid_statuses
    ):
        # Conservative fallback:
        # do not falsely accept an unparseable verification result.
        status = (
            "RETRIEVE_MORE"
        )

        used_fallback = True

    follow_up_queries = []

    for query in parsed.get(
        "follow_up_queries",
        []
    )[
        :max_follow_up_queries
    ]:
        query = " ".join(
            str(query).split()
        )

        if (
            query
            and query
            not in
            follow_up_queries
        ):
            follow_up_queries.append(
                query
            )

    missing_information = []

    for item in parsed.get(
        "missing_information",
        []
    ):
        item = " ".join(
            str(item).split()
        )

        if item:
            missing_information.append(
                item
            )

    verification = {
        "status":
            status,

        "reason":
            str(
                parsed.get(
                    "reason",
                    ""
                )
            ).strip(),

        "missing_information":
            missing_information,

        "follow_up_queries":
            follow_up_queries,

        "raw_output":
            raw_output,

        "used_fallback":
            used_fallback,
    }

    state[
        "verification"
    ] = verification

    state["history"].append({
        "event":
            "verification",

        "iteration":
            state[
                "iteration"
            ],

        "verification":
            verification,
    })

    return state

### Step 13A — Test the Verification Agent

In [ ]:
state_test = verifier_agent(
    state_test
)

print(
    "Verification Status:",
    state_test[
        "verification"
    ]["status"]
)

print(
    "Reason:",
    state_test[
        "verification"
    ]["reason"]
)

print(
    "Missing Information:",
    state_test[
        "verification"
    ][
        "missing_information"
    ]
)

print(
    "Follow-up Queries:",
    state_test[
        "verification"
    ][
        "follow_up_queries"
    ]
)

## Step 14 — Collaboration / Control Loop

This is the component that makes the agents work together.

The loop is:

1. Planning Agent creates the first plan.
2. Retrieval Agent executes its queries.
3. Reranking Agent selects evidence.
4. Answer Agent generates an answer.
5. Verification Agent evaluates the answer/evidence.
6. If `SUPPORTED` → finish.
7. If `RETRIEVE_MORE` → execute verifier follow-up queries and repeat.
8. If `REPLAN` → Planning Agent revises the plan, then retrieval repeats.
9. Stop when supported or the maximum number of iterations is reached.

In [ ]:
def multi_agent_rag(
    question,
    top_k_each=10,
    final_top_k=10,
    max_iterations=3,
    reranker_batch_size=4,
    model=DEEPSEEK_MODEL
):
    state = create_multi_agent_state(
        question
    )

    # --------------------------------------------------------
    # Initial planning
    # --------------------------------------------------------
    state["plan"] = planner_agent(
        question=question,
        model=model
    )

    state["current_queries"] = [
        step["retrieval_query"]
        for step in state[
            "plan"
        ]["steps"]
    ]

    stop_reason = (
        "MAX_ITERATIONS"
    )

    # --------------------------------------------------------
    # Iterative collaboration
    # --------------------------------------------------------
    for iteration in range(
        1,
        max_iterations + 1
    ):
        state["iteration"] = (
            iteration
        )

        # Retrieval Agent
        state = retrieval_agent(
            state=state,
            queries=state[
                "current_queries"
            ],
            top_k_each=top_k_each
        )

        # Reranking Agent
        state = reranking_agent(
            state=state,
            final_top_k=final_top_k,
            batch_size=reranker_batch_size
        )

        # Answer Agent
        state = answer_agent(
            state=state,
            model=model
        )

        # Verification Agent
        state = verifier_agent(
            state=state,
            model=model
        )

        status = (
            state[
                "verification"
            ]["status"]
        )

        # ----------------------------------------------------
        # Successful verification
        # ----------------------------------------------------
        if status == "SUPPORTED":

            stop_reason = (
                "SUPPORTED"
            )

            break

        # ----------------------------------------------------
        # Need more evidence using current plan
        # ----------------------------------------------------
        if status == "RETRIEVE_MORE":

            follow_up_queries = (
                state[
                    "verification"
                ][
                    "follow_up_queries"
                ]
            )

            if follow_up_queries:

                state[
                    "current_queries"
                ] = (
                    follow_up_queries
                )

                continue

            # If verifier requested retrieval but produced no query,
            # revise the plan instead.
            status = "REPLAN"

        # ----------------------------------------------------
        # Replan
        # ----------------------------------------------------
        if status == "REPLAN":

            revised_plan = planner_agent(
                question=question,
                verification_feedback=state[
                    "verification"
                ],
                model=model
            )

            state["plan"] = (
                revised_plan
            )

            state[
                "current_queries"
            ] = [
                step[
                    "retrieval_query"
                ]
                for step
                in revised_plan[
                    "steps"
                ]
            ]

    return {
        "question":
            question,

        "final_answer":
            state[
                "answer"
            ],

        "verification":
            state[
                "verification"
            ],

        "stop_reason":
            stop_reason,

        "iterations_used":
            state[
                "iteration"
            ],

        "final_plan":
            state[
                "plan"
            ],

        "candidate_count":
            len(
                state[
                    "candidate_pool"
                ]
            ),

        "retrieved_passages":
            state[
                "reranked_evidence"
            ],

        "history":
            state[
                "history"
            ],
    }

## Step 15 — Run One Real BeerQA Question End to End

This can make several DeepSeek calls:
- 1 planner call;
- 1 answer call per iteration;
- 1 verifier call per iteration;
- additional planner calls when replanning is requested.

Start with **one question** before running batches.

In [ ]:
test_record = beerqa_sample[0]

multi_agent_test = (
    multi_agent_rag(
        question=test_record[
            "question"
        ],
        max_iterations=3
    )
)

print("QUESTION:")
print(
    test_record[
        "question"
    ]
)

print("\nGOLD ANSWER:")
print(
    test_record[
        "answers"
    ]
)

print("\nMULTI-AGENT ANSWER:")
print(
    multi_agent_test[
        "final_answer"
    ]
)

print(
    "\nSTOP REASON:",
    multi_agent_test[
        "stop_reason"
    ]
)

print(
    "ITERATIONS USED:",
    multi_agent_test[
        "iterations_used"
    ]
)

print(
    "FINAL VERIFICATION:",
    multi_agent_test[
        "verification"
    ]["status"]
)

print(
    "CANDIDATE COUNT:",
    multi_agent_test[
        "candidate_count"
    ]
)

### Step 15A — Inspect the Final Plan

In [ ]:
for step in multi_agent_test[
    "final_plan"
]["steps"]:

    print(
        "Step:",
        step[
            "step_id"
        ]
    )

    print(
        "Need:",
        step[
            "information_need"
        ]
    )

    print(
        "Query:",
        step[
            "retrieval_query"
        ]
    )

    print(
        "Depends on:",
        step[
            "depends_on"
        ]
    )

    print("-" * 80)

### Step 15B — Inspect the Agent Collaboration History

In [ ]:
for event in multi_agent_test[
    "history"
]:
    print("=" * 80)
    print(
        "Event:",
        event[
            "event"
        ]
    )
    print(
        json.dumps(
            event,
            indent=2,
            ensure_ascii=False
        )
    )

## Step 16 — Exact Match and F1 Evaluation

In [ ]:
def normalize_answer(text):
    text = str(text).lower()

    text = "".join(
        ch
        for ch in text
        if ch
        not in string.punctuation
    )

    text = re.sub(
        r"\b(a|an|the)\b",
        " ",
        text
    )

    return " ".join(
        text.split()
    )


def exact_match_score(
    prediction,
    ground_truth
):
    return int(
        normalize_answer(
            prediction
        )
        ==
        normalize_answer(
            ground_truth
        )
    )


def f1_score(
    prediction,
    ground_truth
):
    prediction_tokens = (
        normalize_answer(
            prediction
        ).split()
    )

    ground_truth_tokens = (
        normalize_answer(
            ground_truth
        ).split()
    )

    if (
        not prediction_tokens
        or
        not ground_truth_tokens
    ):
        return float(
            prediction_tokens
            ==
            ground_truth_tokens
        )

    common = (
        Counter(
            prediction_tokens
        )
        &
        Counter(
            ground_truth_tokens
        )
    )

    num_same = sum(
        common.values()
    )

    if num_same == 0:
        return 0.0

    precision = (
        num_same
        /
        len(
            prediction_tokens
        )
    )

    recall = (
        num_same
        /
        len(
            ground_truth_tokens
        )
    )

    return (
        2
        *
        precision
        *
        recall
        /
        (
            precision
            +
            recall
        )
    )


def evaluate_answer(
    prediction,
    gold_answers
):
    if not gold_answers:
        return {
            "em":
                0,

            "f1":
                0.0
        }

    return {
        "em":
            max(
                exact_match_score(
                    prediction,
                    gold
                )
                for gold
                in gold_answers
            ),

        "f1":
            max(
                f1_score(
                    prediction,
                    gold
                )
                for gold
                in gold_answers
            ),
    }

In [ ]:
single_metrics = evaluate_answer(
    prediction=multi_agent_test[
        "final_answer"
    ],
    gold_answers=test_record[
        "answers"
    ]
)

print(
    "Exact Match:",
    single_metrics[
        "em"
    ]
)

print(
    "F1:",
    round(
        single_metrics[
            "f1"
        ],
        4
    )
)

## Step 17 — Batch Evaluation Function

For every record this saves:
- final answer;
- EM / F1;
- final plan;
- verification result;
- stop reason;
- number of iterations;
- candidate count;
- final Top-10 evidence.

The full event history is not saved in the batch result by default because it can make the JSONL very large.

In [ ]:
def evaluate_multi_agent_records(
    records,
    limit=None,
    max_iterations=3
):
    if limit is not None:
        records = records[:limit]

    results = []

    for index, record in enumerate(
        records,
        start=1
    ):
        print(
            f"\nRunning "
            f"{index} / "
            f"{len(records)}"
        )

        print(
            "Question:",
            record[
                "question"
            ]
        )

        try:

            rag_result = (
                multi_agent_rag(
                    question=record[
                        "question"
                    ],
                    max_iterations=max_iterations
                )
            )

            prediction = (
                rag_result[
                    "final_answer"
                ]
            )

            metrics = (
                evaluate_answer(
                    prediction,
                    record[
                        "answers"
                    ]
                )
            )

            retrieved = []

            for passage in rag_result[
                "retrieved_passages"
            ]:

                retrieved.append({
                    "passage_id":
                        passage[
                            "passage_id"
                        ],

                    "title":
                        passage[
                            "title"
                        ],

                    "reranker_score":
                        passage[
                            "reranker_score"
                        ],
                })

            result = {
                "id":
                    record["id"],

                "src":
                    record["src"],

                "question":
                    record[
                        "question"
                    ],

                "gold_answers":
                    record[
                        "answers"
                    ],

                "prediction":
                    prediction,

                "em":
                    metrics["em"],

                "f1":
                    metrics["f1"],

                "stop_reason":
                    rag_result[
                        "stop_reason"
                    ],

                "verification_status":
                    rag_result[
                        "verification"
                    ]["status"],

                "verification_reason":
                    rag_result[
                        "verification"
                    ]["reason"],

                "iterations_used":
                    rag_result[
                        "iterations_used"
                    ],

                "candidate_count":
                    rag_result[
                        "candidate_count"
                    ],

                "final_plan":
                    rag_result[
                        "final_plan"
                    ],

                "retrieved_passages":
                    retrieved,

                "error":
                    None,
            }

            results.append(
                result
            )

            print(
                "Gold:",
                record[
                    "answers"
                ]
            )

            print(
                "Prediction:",
                prediction
            )

            print(
                "EM:",
                metrics[
                    "em"
                ]
            )

            print(
                "F1:",
                round(
                    metrics[
                        "f1"
                    ],
                    4
                )
            )

            print(
                "Iterations:",
                rag_result[
                    "iterations_used"
                ]
            )

            print(
                "Verification:",
                rag_result[
                    "verification"
                ]["status"]
            )

        except Exception as e:

            print(
                "ERROR:",
                e
            )

            results.append({
                "id":
                    record.get(
                        "id"
                    ),

                "src":
                    record.get(
                        "src"
                    ),

                "question":
                    record.get(
                        "question"
                    ),

                "gold_answers":
                    record.get(
                        "answers"
                    ),

                "prediction":
                    None,

                "em":
                    None,

                "f1":
                    None,

                "stop_reason":
                    None,

                "verification_status":
                    None,

                "verification_reason":
                    None,

                "iterations_used":
                    None,

                "candidate_count":
                    None,

                "final_plan":
                    None,

                "retrieved_passages":
                    [],

                "error":
                    str(e),
            })

    return results

## Step 18 — Run Dev5 First

Multi-Agent RAG makes more API calls than the other two pipelines.

For Dev5, each question may use roughly:
- 1 planning call;
- 1–3 answer calls;
- 1–3 verification calls;
- optional replanning calls.

Therefore, always run a small sample first.

In [ ]:
if MA_RAG_DEV5_RESULTS_PATH.exists():

    multi_agent_test5 = []

    with open(
        MA_RAG_DEV5_RESULTS_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:
            multi_agent_test5.append(
                json.loads(line)
            )

    print(
        "Loaded saved Multi-Agent Dev5:",
        len(
            multi_agent_test5
        )
    )

else:

    multi_agent_test5 = (
        evaluate_multi_agent_records(
            records=beerqa_sample,
            limit=5,
            max_iterations=3
        )
    )

    with open(
        MA_RAG_DEV5_RESULTS_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        for result in multi_agent_test5:

            f.write(
                json.dumps(
                    result,
                    ensure_ascii=False
                )
                +
                "\n"
            )

    print(
        "Saved Multi-Agent Dev5:",
        MA_RAG_DEV5_RESULTS_PATH
    )

In [ ]:
successful_ma5 = [
    result
    for result in multi_agent_test5
    if result[
        "error"
    ] is None
]

average_ma5_em = (
    sum(
        result["em"]
        for result
        in successful_ma5
    )
    /
    len(
        successful_ma5
    )
)

average_ma5_f1 = (
    sum(
        result["f1"]
        for result
        in successful_ma5
    )
    /
    len(
        successful_ma5
    )
)

average_ma5_iterations = (
    sum(
        result[
            "iterations_used"
        ]
        for result
        in successful_ma5
    )
    /
    len(
        successful_ma5
    )
)

print(
    "Questions completed:",
    len(
        successful_ma5
    )
)

print(
    "Average EM:",
    round(
        average_ma5_em,
        4
    )
)

print(
    "Average F1:",
    round(
        average_ma5_f1,
        4
    )
)

print(
    "Average iterations:",
    round(
        average_ma5_iterations,
        2
    )
)

## Step 19 — Run Dev20

Only run this after:
- one-question test works;
- agent history looks correct;
- Dev5 works;
- the verifier is not looping unnecessarily.

The Dev20 execution cell is intentionally commented out to avoid accidental API spending.

In [ ]:
if MA_RAG_DEV20_RESULTS_PATH.exists():

    multi_agent_test20 = []

    with open(
        MA_RAG_DEV20_RESULTS_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:
            multi_agent_test20.append(
                json.loads(line)
            )

    print(
        "Loaded saved Multi-Agent Dev20:",
        len(
            multi_agent_test20
        )
    )

else:

    multi_agent_test20 = []

    print(
        "No saved Multi-Agent Dev20 results yet."
    )

    print(
        "Run the next cell intentionally after Dev5 is validated."
    )

In [ ]:
# ------------------------------------------------------------
# RUN THIS ONLY WHEN READY FOR THE FULL DEV20 MULTI-AGENT TEST
# ------------------------------------------------------------

# multi_agent_test20 = (
#     evaluate_multi_agent_records(
#         records=beerqa_sample,
#         limit=DEV_SAMPLE_SIZE,
#         max_iterations=3
#     )
# )
#
# with open(
#     MA_RAG_DEV20_RESULTS_PATH,
#     "w",
#     encoding="utf-8"
# ) as f:
#
#     for result in multi_agent_test20:
#
#         f.write(
#             json.dumps(
#                 result,
#                 ensure_ascii=False
#             )
#             +
#             "\n"
#         )
#
# print(
#     "Saved Multi-Agent results:",
#     MA_RAG_DEV20_RESULTS_PATH
# )

In [ ]:
if multi_agent_test20:

    successful_ma20 = [
        result
        for result
        in multi_agent_test20
        if result[
            "error"
        ] is None
    ]

    average_ma20_em = (
        sum(
            result["em"]
            for result
            in successful_ma20
        )
        /
        len(
            successful_ma20
        )
    )

    average_ma20_f1 = (
        sum(
            result["f1"]
            for result
            in successful_ma20
        )
        /
        len(
            successful_ma20
        )
    )

    average_ma20_iterations = (
        sum(
            result[
                "iterations_used"
            ]
            for result
            in successful_ma20
        )
        /
        len(
            successful_ma20
        )
    )

    supported_count = sum(
        1
        for result
        in successful_ma20
        if result[
            "verification_status"
        ] == "SUPPORTED"
    )

    print(
        "Total questions:",
        len(
            multi_agent_test20
        )
    )

    print(
        "Successful:",
        len(
            successful_ma20
        )
    )

    print(
        "Errors:",
        len(
            multi_agent_test20
        )
        -
        len(
            successful_ma20
        )
    )

    print(
        "Average EM:",
        round(
            average_ma20_em,
            4
        )
    )

    print(
        "Average F1:",
        round(
            average_ma20_f1,
            4
        )
    )

    print(
        "Average iterations:",
        round(
            average_ma20_iterations,
            2
        )
    )

    print(
        "Verifier SUPPORTED:",
        supported_count,
        "/",
        len(
            successful_ma20
        )
    )

## Step 20 — Development Retrieval Metrics

Use the same title-level development metrics as the Standard and Query Decomposition notebooks:
- Title Recall@10
- Title Precision@10

These are development diagnostics, not final passage-level BeerQA metrics.

In [ ]:
def get_gold_titles(record):
    gold_titles = []

    for item in record.get(
        "context",
        []
    ):
        if (
            isinstance(
                item,
                list
            )
            and
            len(item) >= 1
        ):
            title = normalize_title(
                item[0]
            )

            if (
                title
                and
                title
                not in gold_titles
            ):
                gold_titles.append(
                    title
                )

    return gold_titles


def get_retrieved_titles(
    result,
    top_k=10
):
    return [
        normalize_title(
            passage[
                "title"
            ]
        )
        for passage
        in result[
            "retrieved_passages"
        ][:top_k]
    ]


def title_recall_at_k(
    retrieved_titles,
    gold_titles
):
    gold_set = set(
        gold_titles
    )

    if not gold_set:
        return 0.0

    matched = (
        set(
            retrieved_titles
        )
        &
        gold_set
    )

    return (
        len(matched)
        /
        len(gold_set)
    )


def title_precision_at_k(
    retrieved_titles,
    gold_titles
):
    unique_retrieved = list(
        dict.fromkeys(
            retrieved_titles
        )
    )

    if not unique_retrieved:
        return 0.0

    matched = (
        set(
            unique_retrieved
        )
        &
        set(
            gold_titles
        )
    )

    return (
        len(matched)
        /
        len(
            unique_retrieved
        )
    )

In [ ]:
if multi_agent_test20:

    retrieval_rows = []

    for record, result in zip(
        beerqa_sample,
        multi_agent_test20
    ):
        gold_titles = (
            get_gold_titles(
                record
            )
        )

        retrieved_titles = (
            get_retrieved_titles(
                result,
                top_k=10
            )
        )

        recall10 = (
            title_recall_at_k(
                retrieved_titles,
                gold_titles
            )
        )

        precision10 = (
            title_precision_at_k(
                retrieved_titles,
                gold_titles
            )
        )

        result["gold_titles"] = (
            gold_titles
        )

        result[
            "title_recall@10"
        ] = recall10

        result[
            "title_precision@10"
        ] = precision10

        retrieval_rows.append({
            "title_recall@10":
                recall10,

            "title_precision@10":
                precision10,
        })

    average_ma_recall10 = (
        sum(
            row[
                "title_recall@10"
            ]
            for row
            in retrieval_rows
        )
        /
        len(
            retrieval_rows
        )
    )

    average_ma_precision10 = (
        sum(
            row[
                "title_precision@10"
            ]
            for row
            in retrieval_rows
        )
        /
        len(
            retrieval_rows
        )
    )

    print(
        "Average Title Recall@10:",
        round(
            average_ma_recall10,
            4
        )
    )

    print(
        "Average Title Precision@10:",
        round(
            average_ma_precision10,
            4
        )
    )

## Step 21 — Save Fully Evaluated Multi-Agent Dev20 Results

In [ ]:
if multi_agent_test20:

    with open(
        MA_RAG_DEV20_EVAL_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        for result in multi_agent_test20:

            f.write(
                json.dumps(
                    result,
                    ensure_ascii=False
                )
                +
                "\n"
            )

    print(
        "Saved evaluated Multi-Agent results:",
        MA_RAG_DEV20_EVAL_PATH
    )

## Step 22 — Compare the Three Development Pipelines

This section compares saved results when available:

- Standard RAG
- Query Decomposition RAG
- Multi-Agent RAG

In [ ]:
def load_jsonl(path):
    rows = []

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:
            rows.append(
                json.loads(line)
            )

    return rows


def summarize_results(rows):
    successful = [
        row
        for row in rows
        if row.get(
            "error"
        ) is None
    ]

    summary = {
        "n":
            len(
                successful
            ),

        "em":
            sum(
                row["em"]
                for row
                in successful
            )
            /
            len(
                successful
            ),

        "f1":
            sum(
                row["f1"]
                for row
                in successful
            )
            /
            len(
                successful
            ),
    }

    if all(
        "title_recall@10"
        in row
        for row
        in successful
    ):
        summary[
            "title_recall@10"
        ] = (
            sum(
                row[
                    "title_recall@10"
                ]
                for row
                in successful
            )
            /
            len(
                successful
            )
        )

    if all(
        "title_precision@10"
        in row
        for row
        in successful
    ):
        summary[
            "title_precision@10"
        ] = (
            sum(
                row[
                    "title_precision@10"
                ]
                for row
                in successful
            )
            /
            len(
                successful
            )
        )

    return summary

In [ ]:
available = {}

if STANDARD_RAG_EVAL_PATH.exists():
    available[
        "Standard RAG"
    ] = summarize_results(
        load_jsonl(
            STANDARD_RAG_EVAL_PATH
        )
    )

if QD_RAG_EVAL_PATH.exists():
    available[
        "Query Decomposition"
    ] = summarize_results(
        load_jsonl(
            QD_RAG_EVAL_PATH
        )
    )

if MA_RAG_DEV20_EVAL_PATH.exists():
    available[
        "Multi-Agent RAG"
    ] = summarize_results(
        load_jsonl(
            MA_RAG_DEV20_EVAL_PATH
        )
    )

if available:

    print("=" * 100)

    print(
        f"{'Pipeline':<24}"
        f"{'EM':>12}"
        f"{'F1':>12}"
        f"{'Recall@10':>16}"
        f"{'Precision@10':>18}"
    )

    print("=" * 100)

    for name, summary in available.items():

        print(
            f"{name:<24}"
            f"{summary.get('em', float('nan')):>12.4f}"
            f"{summary.get('f1', float('nan')):>12.4f}"
            f"{summary.get('title_recall@10', float('nan')):>16.4f}"
            f"{summary.get('title_precision@10', float('nan')):>18.4f}"
        )

else:

    print(
        "No evaluated result files are available yet."
    )

# Completion Checklist

The Multi-Agent RAG development pipeline is complete when you have verified:

- [ ] Planning Agent creates sensible information needs and retrieval queries.
- [ ] Retrieval Agent executes BM25 + DPR/FAISS for planner queries.
- [ ] Candidate evidence accumulates across iterations without duplicate passage IDs.
- [ ] Reranking Agent selects the final Top-10 evidence.
- [ ] Answer Agent generates a grounded answer.
- [ ] Verification Agent returns `SUPPORTED`, `RETRIEVE_MORE`, or `REPLAN`.
- [ ] `RETRIEVE_MORE` causes new retrieval.
- [ ] `REPLAN` causes the Planning Agent to revise the plan.
- [ ] The collaboration loop stops at `SUPPORTED` or `max_iterations`.
- [ ] One real BeerQA question works end to end.
- [ ] Dev5 works.
- [ ] Dev20 works.
- [ ] EM / F1 are calculated.
- [ ] Title Recall@10 / Title Precision@10 are calculated.
- [ ] Evaluated results are saved.
- [ ] Results can be compared with Standard RAG and Query Decomposition RAG.

After this, the next major component is the **DeepSeek complexity-aware router** that chooses:

- Easy → Standard RAG
- Medium → Query Decomposition RAG
- Hard → Multi-Agent RAG